# Đề bài về nhà

## Yêu cầu

- Tự viết code cho mô hình Linear Regression theo công thức đã được dạy trong buổi lý thuyết trên lớp.
- Tự viết hàm dự đoán.
- Huấn luyện cả mô hình của thư viện và mô hình mình tự viết.
- In ra các trọng số: w0, w1, w2, ..., wn của cả 2 mô hình đã huấn luyện để quan sát và so sánh.
- Dự đoán dữ liệu tập test bằng cả 2 mô hình (mô hình thư viện thì dùng hàm predict() của thư viện, mô hình tự viết dùng hàm dự đoán tự viết), in ra kết quả bằng Dataframe như trong bài thực hành trên lớp.
- Tính RMSE trên tập test cho cả 2 mô hình và so sánh.

## Dữ liệu

Tập dữ liệu giá nhà ở Boston đã có sẵn trên sklearn, dữ liệu đã được chuẩn hóa và chia thành tập train, tập test

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math

from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score

# Đọc dữ liệu

Dữ liệu về giá nhà ở Boston được hỗ trợ bởi sklearn, đọc dữ liệu thông qua hàm `datasets.load_boston()`

Xem thêm các bộ dữ liệu khác tại https://scikit-learn.org/stable/datasets/index.html#toy-datasets.

Dữ liệu được chia thành các thành phần data và target như tập diabetes. Dữ liệu cũng đã được chuẩn hóa, chỉ cần gọi ra và huấn luyện

In [ ]:
# Load data from the new URL as requested
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

print("Số chiều dữ liệu input: ", data.shape)
print("Số chiều dữ liệu target: ", target.shape)
print("\n5 mẫu dữ liệu đầu tiên:")
print("input: ", data[:5])
print("target: ", target[:5])

# Gán lại vào biến dataset để tương thích với các bước sau
class Dataset:
    def __init__(self, data, target):
        self.data = data
        self.target = target
dataset = Dataset(data, target)

Số chiều dữ liệu input:  (506, 13)
Số chiều dữ liệu target:  (506,)

5 mẫu dữ liệu đầu tiên:
input:  [[6.3200e-03 1.8000e+01 2.3100e+00 0.0000e+00 5.3800e-01 6.5750e+00
  6.5200e+01 4.0900e+00 1.0000e+00 2.9600e+02 1.5300e+01 3.9690e+02
  4.9800e+00]
 [2.7310e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 6.4210e+00
  7.8900e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9690e+02
  9.1400e+00]
 [2.7290e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 7.1850e+00
  6.1100e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9283e+02
  4.0300e+00]
 [3.2370e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 6.9980e+00
  4.5800e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9463e+02
  2.9400e+00]
 [6.9050e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 7.1470e+00
  5.4200e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9690e+02
  5.3300e+00]]
target:  [24.  21.6 34.7 33.4 36.2]


**Chia dữ liệu làm 2 phần training 362 mẫu và testing 80 mẫu**

In [ ]:
# Chia dữ liệu thành tập train và test
dataset_X = dataset.data
dataset_y = dataset.target

# Chia theo tỷ lệ train: 404 mẫu, test: phần còn lại
dataset_X_train = dataset_X[:404]
dataset_y_train = dataset_y[:404]

dataset_X_test = dataset_X[404:]
dataset_y_test = dataset_y[404:]

print("Kích thước tập train:", dataset_X_train.shape)
print("Kích thước tập test:", dataset_X_test.shape)

Kích thước tập train: (404, 13)
Kích thước tập test: (102, 13)


# Xây dựng mô hình

In [ ]:
# 1. Xây dựng mô hình bằng thư viện
from sklearn.linear_model import LinearRegression
model_lib = LinearRegression()

# 2. Xây dựng mô hình tự viết (Normal Equation)
class MyLinearRegression:
    def __init__(self):
        self.w = None

    def fit(self, X, y):
        # Thêm cột 1 (bias term)
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        # Công thức: w = (X^T * X)^-1 * X^T * y
        self.w = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.w)

## Xây dựng mô hình bằng thư viện

In [ ]:
# Huấn luyện mô hình bằng thư viện sklearn
from sklearn.linear_model import LinearRegression
model_lib = LinearRegression()
model_lib.fit(dataset_X_train, dataset_y_train)

LinearRegression()

## Xây dựng mô hình Linear Regression tự viết

In [ ]:
# Xây dựng lớp Linear Regression tự viết sử dụng Normal Equation
class MyLinearRegression:
    def __init__(self):
        self.w = None

    def fit(self, X, y):
        # Thêm cột 1 (bias term)
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        # Công thức: w = (X^T * X)^-1 * X^T * y
        self.w = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.w)

## Hàm test mô hình tự viết

In [ ]:
# Khởi tạo và huấn luyện thử mô hình tự viết
my_model = MyLinearRegression()
my_model.fit(dataset_X_train, dataset_y_train)
print("Đã huấn luyện xong mô hình tự viết.")

Đã huấn luyện xong mô hình tự viết.


# Huấn luyện mô hình

In [ ]:
# Huấn luyện mô hình thư viện
model_lib.fit(dataset_X_train, dataset_y_train)
print("--- Trọng số mô hình thư viện ---")
print("w0 (Intercept):", model_lib.intercept_)
print("w1...wn (Coefficients):", model_lib.coef_)

# Huấn luyện mô hình tự viết
my_model = MyLinearRegression()
my_model.fit(dataset_X_train, dataset_y_train)
print("\n--- Trọng số mô hình tự viết ---")
print("Weights (w0, w1...wn):", my_model.w)

--- Trọng số mô hình thư viện ---
w0 (Intercept): 30.077166922901817
w1...wn (Coefficients): [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]

--- Trọng số mô hình tự viết ---
Weights (w0, w1...wn): [ 3.00771669e+01 -2.02135297e-01  4.41276341e-02  5.26739364e-02
  1.88474315e+00 -1.49281487e+01  4.76038673e+00  2.88734527e-03
 -1.30025278e+00  4.61661953e-01 -1.55434673e-02 -8.11632369e-01
 -1.97174433e-03 -5.32273431e-01]


## Huấn luyện mô hình của thư viện

In [ ]:
# In trọng số của mô hình thư viện
print("--- Trọng số mô hình thư viện ---")
print("w0 (Intercept):", model_lib.intercept_)
print("w1...wn (Coefficients):", model_lib.coef_)

--- Trọng số mô hình thư viện ---
w0 (Intercept): 30.077166922901817
w1...wn (Coefficients): [-2.02135297e-01  4.41276341e-02  5.26739364e-02  1.88474315e+00
 -1.49281487e+01  4.76038673e+00  2.88734527e-03 -1.30025278e+00
  4.61661953e-01 -1.55434673e-02 -8.11632369e-01 -1.97174433e-03
 -5.32273431e-01]


## Training mô hình bằng Linear regression tự viết

In [ ]:
# In trọng số của mô hình tự viết để so sánh
print("--- Trọng số mô hình tự viết ---")
print("Weights (w0, w1...wn):", my_model.w)

--- Trọng số mô hình tự viết ---
Weights (w0, w1...wn): [ 3.00771669e+01 -2.02135297e-01  4.41276341e-02  5.26739364e-02
  1.88474315e+00 -1.49281487e+01  4.76038673e+00  2.88734527e-03
 -1.30025278e+00  4.61661953e-01 -1.55434673e-02 -8.11632369e-01
 -1.97174433e-03 -5.32273431e-01]


# Dự đoán các mẫu dữ liệu

In [ ]:
# Dự đoán trên tập test bằng cả 2 mô hình
y_pred_lib = model_lib.predict(dataset_X_test)
y_pred_my = my_model.predict(dataset_X_test)

# In kết quả bằng Dataframe
comparison_df = pd.DataFrame({
    'Giá thực tế': dataset_y_test,
    'Dự đoán (Thư viện)': y_pred_lib,
    'Dự đoán (Tự viết)': y_pred_my
})
display(comparison_df.head(10))

,Giá thực tế,Dự đoán (Thư viện),Dự đoán (Tự viết)
0,8.5,5.886799,5.886799
1,5.0,3.787057,3.787057
2,11.9,6.640550,6.640550
3,27.9,21.312765,21.312765
4,17.2,15.412714,15.412714
5,27.5,23.652298,23.652298
6,15.0,16.585687,16.585687
7,17.2,22.239823,22.239823
8,17.9,4.597394,4.597394
9,16.3,12.316953,12.316953


## Dự đoán các mẫu dữ liệu theo mô hình của thư viện

In [ ]:
# Dự đoán trên tập test bằng mô hình thư viện
y_pred_lib = model_lib.predict(dataset_X_test)
print("Dự đoán xong bằng mô hình thư viện.")

Dự đoán xong bằng mô hình thư viện.


## Dự đoán các mẫu dữ liệu tính theo linear regression tự viết

In [ ]:
# Dự đoán trên tập test bằng mô hình tự viết và hiển thị so sánh
y_pred_my = my_model.predict(dataset_X_test)

comparison_df = pd.DataFrame({
    'Giá thực tế': dataset_y_test,
    'Dự đoán (Thư viện)': y_pred_lib,
    'Dự đoán (Tự viết)': y_pred_my
})
display(comparison_df.head(10))

,Giá thực tế,Dự đoán (Thư viện),Dự đoán (Tự viết)
0,5.0,3.787057,3.787057
1,11.9,6.640550,6.640550
2,27.9,21.312765,21.312765
3,17.2,15.412714,15.412714
4,27.5,23.652298,23.652298
5,15.0,16.585687,16.585687
6,17.2,22.239823,22.239823
7,17.9,4.597394,4.597394
8,16.3,12.316953,12.316953
9,7.0,-4.441254,-4.441254


## Đánh giá mô hình linear regression của thư viện

In [ ]:
# Đánh giá RMSE cho cả hai mô hình
from sklearn.metrics import mean_squared_error

rmse_lib = np.sqrt(mean_squared_error(dataset_y_test, y_pred_lib))
rmse_my = np.sqrt(mean_squared_error(dataset_y_test, y_pred_my))

print(f"RMSE Mô hình thư viện: {rmse_lib}")
print(f"RMSE Mô hình tự viết: {rmse_my}")
print("\nSo sánh: Sai số của hai mô hình là tương đương nhau.")

RMSE Mô hình thư viện: 5.749521870254049
RMSE Mô hình tự viết: 5.749521870254732

So sánh: Sai số của hai mô hình là tương đương nhau.


## Đánh giá mô hình linear regression tự viết

In [ ]:
# Tính RMSE cho cả 2 mô hình
from sklearn.metrics import mean_squared_error
rmse_lib = np.sqrt(mean_squared_error(dataset_y_test, y_pred_lib))
rmse_my = np.sqrt(mean_squared_error(dataset_y_test, y_pred_my))

print(f"RMSE Mô hình thư viện: {rmse_lib}")
print(f"RMSE Mô hình tự viết: {rmse_my}")

# Nhận xét
print("\nNhận xét: Hai mô hình cho kết quả tương đồng vì cùng sử dụng phương pháp bình phương tối thiểu (OLS).")

RMSE Mô hình thư viện: 5.727116436760027
RMSE Mô hình tự viết: 5.727116436760692

Nhận xét: Hai mô hình cho kết quả tương đồng vì cùng sử dụng phương pháp bình phương tối thiểu (OLS).
